# **Machine Learning**

In this section, we build and evaluate machine learning models to classify brain MRI images into four categories: glioma, meningioma, no tumor, and pituitary tumor.

Each image is converted into a numerical feature vector and used as input for several machine learning algorithms. Models are trained, compared, and evaluated to identify the most suitable approach for this classification task.

---

## **Methodology**

The approach followed in this project is structured in the following steps:

1. Converting images into **feature vectors** (flattened pixel arrays)
2. Training multiple machine learning models on the training set
3. Comparing their performance using accuracy and other metrics
4. Selecting the best model and tuning its hyperparameters
5. Evaluating the final model on the test set
6. Look how we could upgrade the model with some graph with more details

---

## **Model Comparison**

Before selecting a final model, we compare four different algorithms to identify which approach is best suited for our dataset.

| Model | Type | Description |
|---|---|---|
| **Logistic Regression** | Linear | Simple baseline model |
| **Decision Tree** | Non-linear | Captures complex relationships |
| **Random Forest** | Ensemble | Combines multiple decision trees |
| **KNN** | Distance-based | Classifies by similarity |

In [ ]:
from pathlib import Path
import sys

# Auto-detect project root — works regardless of where Jupyter was launched
# Walks up from CWD until it finds the directory containing supervised/
_search = Path.cwd()
while not (_search / "supervised").exists() and _search != _search.parent:
    _search = _search.parent

PROJECT_ROOT   = _search
SUPERVISED_ROOT = PROJECT_ROOT / "supervised"
CSV_PATH       = str(SUPERVISED_ROOT / "dataset" / "dataset_info.csv")
DATASET_DIR    = str(SUPERVISED_ROOT / "dataset")

# Make supervised package importable from anywhere
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root : {PROJECT_ROOT}")
print(f"CSV          : {CSV_PATH}")

## **Configuration — Fast Mode**

Training time depends heavily on image resolution. By default the pipeline runs at **224×224** (full quality). Set `FAST_MODE = True` to switch to **32×32** for quick prototyping.

| Mode | Resolution | Features | Data loading | Model comparison | RandomizedSearchCV (n_iter=50) |
|------|-----------|----------|--------------|-----------------|-------------------------------|
| **Full** | 224×224 | 150 528 | ~15 min | ~30–60 min | several hours |
| **Medium** | 64×64 | 12 288 | ~5 min | ~10–20 min | ~30–60 min |
| **Fast** | 32×32 | 3 072 | ~3 min | ~3–5 min | ~10–15 min |

> **Note:** Fast mode reduces accuracy slightly (32×32 loses fine detail) but is useful to verify the pipeline end-to-end before a full run.

In [ ]:
# ─── Configuration ────────────────────────────────────────────────────────────
# Set FAST_MODE = True for quick prototyping (~3 min load, ~5 min training)
# Set FAST_MODE = False for full quality run (~15 min load, hours of training)

FAST_MODE = True

if FAST_MODE:
    target_size = (32, 32)
    n_iter_search = 20          # fewer hyperparameter combinations
    print("FAST MODE — 32×32 pixels, n_iter=20")
    print("  Data loading : ~3 min")
    print("  Model comparison : ~3–5 min")
    print("  RandomizedSearchCV : ~10–15 min")
else:
    target_size = (224, 224)
    n_iter_search = 200
    print("FULL MODE — 224×224 pixels, n_iter=200")
    print("  Data loading : ~15 min")
    print("  Model comparison : ~30–60 min")
    print("  RandomizedSearchCV : several hours")
    print("  Tip: set FAST_MODE = True above to run faster")

In [ ]:
import time
import joblib
import matplotlib.pyplot as plt

from supervised.src.utils import load_data
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score


def compare_models(X_train, y_train, X_test, y_test):
    models = {
        "RandomForest": RandomForestClassifier(),
        "LogisticRegression": LogisticRegression(max_iter=1000),
        "KNN": KNeighborsClassifier(n_neighbors=5),
        "DecisionTree": DecisionTreeClassifier()
    }

    res = {}
    timings = {}

    for name, model in models.items():
        t0 = time.time()
        model.fit(X_train, y_train)
        elapsed = time.time() - t0

        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)

        res[name] = acc
        timings[name] = elapsed
        print(f"{name:22s}  accuracy={acc:.3f}  train_time={elapsed:.1f}s")

    return res, timings

def plot_model_comparison(results, timings=None):
    names = list(results.keys())
    values = list(results.values())

    if timings:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    else:
        fig, ax1 = plt.subplots(figsize=(8, 5))

    bars = ax1.bar(names, values)
    for bar, value in zip(bars, values):
        ax1.text(bar.get_x() + bar.get_width() / 2, value, f"{value:.2f}", ha='center', va='bottom')
    ax1.set_ylim(0, 1)
    ax1.set_title("Model Comparison (Accuracy)")
    ax1.set_ylabel("Accuracy")

    if timings:
        t_names = list(timings.keys())
        t_values = list(timings.values())
        bars2 = ax2.bar(t_names, t_values, color="steelblue")
        for bar, value in zip(bars2, t_values):
            ax2.text(bar.get_x() + bar.get_width() / 2, value, f"{value:.1f}s", ha='center', va='bottom')
        ax2.set_title("Model Training Time (seconds)")
        ax2.set_ylabel("Time (s)")

    plt.tight_layout()
    plt.show()

X_train, y_train, X_test, y_test = load_data(CSV_PATH, DATASET_DIR, target_size=target_size)
results, timings = compare_models(X_train, y_train, X_test, y_test)
plot_model_comparison(results, timings)

## **Model Comparison Results**

| Model | Accuracy |
|---|----------|
| **Random Forest** | **93%**  |
| Logistic Regression | 90%      |
| Decision Tree | 87%      |
| KNN | 85%      |

Random Forest achieves the highest accuracy at **93%**, outperforming all other models. It is therefore selected as the final model for hyperparameter tuning.

---

## **Model Comparison Conclusion**

Random Forest achieves the best accuracy at **93%**, outperforming all other tested models. As an ensemble method combining multiple decision trees, it is particularly well suited for capturing complex and non-linear patterns in image data.

Logistic Regression performs surprisingly well at **90%**, showing that even a linear model can extract useful information from pixel values.<br>
Decision Tree and KNN follow at **87%** and **85%** respectively, limited by their tendency to overfit or their sensitivity to high-dimensional data.

**Random Forest is therefore selected as the final model** for hyperparameter tuning.

---

## **Hyperparameter Tuning**

Hyperparameters are parameters that are not learned by the model during training, but set before training begins.<br>
They control the behavior and complexity of the model, and can significantly impact its ability to generalize to unseen data.

Unlike model parameters which are optimized automatically during training, hyperparameters must be tuned manually or through search strategies such as **RandomizedSearchCV**, which tests multiple combinations of parameters and selects the best one based on cross-validation performance.

The following hyperparameters are explored for Random Forest:

| Hyperparameter | Values tested | Role |
|---|---|---|
| `n_estimators` | 100, 200, 300 | Number of trees — more trees increases stability and precision |
| `max_depth` | 3, 5, 7 | Maximum depth — low values risk underfitting, high values risk overfitting |
| `min_samples_split` | 15, 20, 30 | Minimum samples to split a node — limits unnecessary divisions |
| `min_samples_leaf` | 8, 12, 16 | Minimum samples per leaf — forces leaves to contain multiple examples |
| `max_features` | 0.3, 0.5, sqrt | Features considered per split — controls diversity between trees |
| `bootstrap` | True | Whether to use bootstrap samples when building trees |

In [ ]:
import time
import joblib
import os

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV


def tune_random_forest(X_train, y_train, X_test, y_test, n_iter=200):
    param_dist = {
        "n_estimators": [100, 200, 300],
        "max_depth": [3, 5, 7],
        "min_samples_split": [15, 20, 30],
        "min_samples_leaf": [8, 12, 16],
        "bootstrap": [True],
        "max_features": [0.3, 0.5, "sqrt"],
    }

    model = RandomForestClassifier(random_state=42, oob_score=True, n_jobs=-1)

    search = RandomizedSearchCV(
        model,
        param_distributions=param_dist,
        n_iter=n_iter,
        cv=5,
        scoring="accuracy",
        random_state=42,
        n_jobs=-1,
        verbose=1
    )

    t0 = time.time()
    search.fit(X_train, y_train)
    elapsed = time.time() - t0

    best_model = search.best_estimator_

    print(f"Search time: {elapsed:.1f}s  ({elapsed/60:.1f} min)")
    print("Best parameters:", search.best_params_)
    print("Best CV score:", search.best_score_)
    print("Best OOB score:", best_model.oob_score_)
    print("Test accuracy:", best_model.score(X_test, y_test))

    return best_model

best_model = tune_random_forest(X_train, y_train, X_test, y_test, n_iter=n_iter_search)

model_dir = str(SUPERVISED_ROOT / "model")
os.makedirs(model_dir, exist_ok=True)
joblib.dump(best_model, str(SUPERVISED_ROOT / "model" / "best_model.pkl"))
print(f"Model saved to {SUPERVISED_ROOT / 'model' / 'best_model.pkl'}")

## **Final Model**

The final model is a **Random Forest** with the following parameters:

| Hyperparameter | Value |
|---|---|
| `n_estimators` | 300 |
| `max_depth` | None |
| `min_samples_split` | 20 |
| `min_samples_leaf` | 8 |
| `max_features` | sqrt |
| `criterion` | gini |
| `bootstrap` | True |

| Metric | Value |
|---|---|
| Train Accuracy | 95.6% |
| Test Accuracy | 86.4% |
| Gap | 9.1% |

---

## **Conclusion**

Among the four models tested, **Random Forest** achieved the best
performance with a test accuracy of **94%** before tuning, and **86.4%**
after hyperparameter optimization on the full search space.

The model shows a gap of **9.1%** between train and test accuracy,
indicating slight overfitting. This is partly due to `max_depth=None`
which allows trees to grow without any limit.

For a medical application such as brain tumor detection, accuracy alone
is not sufficient. Future improvements could include:
- Constraining `max_depth` to reduce overfitting
- Using a CNN architecture better suited for image classification
- Adding data augmentation to improve generalization

---

## **Confusion Matrix**

Accuracy alone is not sufficient to evaluate a model on a medical dataset.<br>
A model could achieve high overall accuracy while performing poorly on a specific cancer type, which would be particularly dangerous in a real-world diagnostic context.

The confusion matrix allows us to visualize the performance of the model for each individual class, showing not only how many samples were correctly
classified, but also which classes are most frequently confused with each other.

In [ ]:
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

def plot_confusion_matrix(model, X_test, y_test):
    y_pred = model.predict(X_test)

    cm = confusion_matrix(y_test, y_pred)
    class_names = ["Glioma", "Meningioma", "No Tumor", "Pituitary"]

    fig, ax = plt.subplots(figsize=(8, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(ax=ax, cmap="Blues")

    ax.set_title("Confusion Matrix")
    plt.tight_layout()
    plt.show()

plot_confusion_matrix(best_model, X_test, y_test)

## **Confusion Matrix Analysis**

The confusion matrix reveals significant differences in performance
across the four classes:

- **No Tumor (98%)** and **Pituitary (96%)** are classified with high accuracy, suggesting strong visual distinction from other classes.<br>
- **Meningioma (67%)** is the most difficult class to classify, frequently confused with No Tumor (69 cases). This suggests that meningioma images share visual similarities with no tumor images, making them harder to distinguish for the model.<br>
- **Glioma (79%)** shows moderate performance, with some confusion with Meningioma (53 cases).

In a medical context, these misclassifications are critical, as confusing two cancer types or missing a tumor entirely could have serious consequences for patient diagnosis.

---

## **Classification Report**

Accuracy alone gives a single global score for the entire model, but does not reveal how the model performs on each individual class.<br>
A model could achieve high overall accuracy while completely failing on a specific cancer type, which would be unacceptable in a medical diagnostic context.

The classification report addresses this limitation by providing three detailed metrics for each class: precision, recall, and F1-score.<br>
Together, these metrics give a much more complete picture of the model's strengths and weaknesses across all four cancer types.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from sklearn.metrics import classification_report

def plot_classification_report(model, X_test, y_test):
    y_pred = model.predict(X_test)

    class_names = ["Glioma", "Meningioma", "No Tumor", "Pituitary"]
    report = classification_report(y_test, y_pred, target_names=class_names, output_dict=True)
    metrics = ["precision", "recall", "f1-score"]
    x = np.arange(len(class_names))
    width = 0.25

    fig, ax = plt.subplots(figsize=(12, 6))

    for i, metric in enumerate(metrics):
        values = [report[cls][metric] for cls in class_names]
        bars = ax.bar(x + i * width, values, width, label=metric.capitalize())
        for bar, value in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{value:.2f}", ha="center", va="bottom", fontsize=8)

    ax.set_xticks(x + width)
    ax.set_xticklabels(class_names)
    ax.set_ylim(0, 1.1)
    ax.set_ylabel("Score")
    ax.set_title("Classification Report per Class")
    ax.legend()
    ax.grid(axis="y", linestyle="--", alpha=0.7)

    plt.tight_layout()
    plt.show()

plot_classification_report(best_model, X_test, y_test)

### Results Interpretation

Before analyzing the results, here is a brief explanation of each metric:

- **Precision**: among all samples predicted as a given class, how many were actually correct. A high precision means few false alarms.
- **Recall**: among all real samples of a given class, how many were correctly detected. A high recall means few missed cases.
- **F1-Score**: the harmonic mean of precision and recall. It provides a single balanced metric, particularly useful when the two metrics are in conflict.

In a medical context, **recall is the most critical metric**.<br>
Missing a real tumor (low recall) is far more dangerous than a false alarm (low precision).

| Class | Precision | Recall | F1-Score |
|---|---|---|---|
| Glioma | 0.93 | 0.79 | 0.85 |
| Meningioma | 0.77 | 0.68 | 0.72 |
| No Tumor | 0.85 | 0.99 | 0.91 |
| Pituitary | 0.91 | 0.96 | 0.94 |

**No Tumor** and **Pituitary** achieve the best results with F1-scores
of **0.91** and **0.94** respectively, confirming that these classes
are the easiest to classify.

**Meningioma** remains the most challenging class with the lowest
F1-score of **0.72**. Its recall of **0.68** means that **32% of real
meningioma cases are missed** by the model, which is particularly
concerning in a medical context.

**Glioma** shows high precision (**0.93**) but lower recall (**0.79**),
meaning the model is conservative when predicting glioma but still
misses some real cases.

Overall, the model achieves a weighted F1-score of **0.86**, consistent
with the global test accuracy of **86.4%**.